# Conservación de la energía generalizada según el paso

Para un Hamiltoniano dependiente del tiempo se comprueba la conservación de $K(t)=H(t,z(t))+k(t)$ en GC. Al final se ejecuta también una prueba corta de la variante FC.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from classes import Potential, TrajectoryGC, SystemGC, TrajectoryFC, SystemFC


def generalized_energy(system, solution):
    hamiltonian = np.asarray(system.hamiltonian(solution.t, solution.y))
    return hamiltonian + np.asarray(solution.k)

In [ ]:
potential = Potential.random(
    A=0.7,
    M=25,
    nx=64,
    ny=64,
    seed=27,
    interpolation_order=5,
)
x0 = potential.grid.xmin + potential.grid.period / 2
y0 = potential.grid.ymin + potential.grid.period / 2

gc_trajectory = TrajectoryGC(rho=0.3)
gc_trajectory.set_initial_state(np.array([x0, y0]))
gc_system = SystemGC(potential, gc_trajectory)

In [ ]:
steps = (0.004, 0.001, 0.00025)
t_span = (0.0, 6 * np.pi)
n_save_step = 361
solutions = {}
relative_errors = {}

for integration_step in steps:
    solution = gc_system.simulate(
        step=integration_step,
        t_span=t_span,
        n_save_step=n_save_step,
        check_energy=True,
        progress=True,
    )
    energy = generalized_energy(gc_system, solution)[0]
    energy_scale = max(abs(energy[0]), np.finfo(float).eps)
    relative_error = (energy - energy[0]) / energy_scale
    solutions[integration_step] = solution
    relative_errors[integration_step] = relative_error

print(f"{'step':>10} {'n_steps':>12} {'sol.err':>16} {'max |error rel.|':>20}")
for integration_step in steps:
    solution = solutions[integration_step]
    max_error = np.max(np.abs(relative_errors[integration_step]))
    print(
        f"{integration_step:10.4g} {solution.n_steps:12d} "
        f"{solution.err:16.8e} {max_error:20.8e}"
    )

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5), constrained_layout=True)

for integration_step in steps:
    solution = solutions[integration_step]
    error = relative_errors[integration_step]
    ax.plot(
        solution.t,
        error,
        label=(
            fr'$\Delta t={integration_step:g}$'
            fr' — $\max|\varepsilon_K|={np.max(np.abs(error)):.3e}$'
        ),
    )

ax.axhline(0.0, color='0.5', ls='--', lw=1)
ax.set(
    xlabel='$t$',
    ylabel=r'$\varepsilon_K(t)=(K(t)-K(0))/|K(0)|$',
    title=r'Conservación GC de la energía generalizada $K=H+k$',
)
ax.ticklabel_format(axis='y', style='sci', scilimits=(0, 0))
ax.grid(alpha=0.25)
ax.legend()
plt.show()

## Prueba corta FC

Una integración corta verifica el estado FC en bloques `[x, y, vx, vy]` y su energía generalizada.

In [ ]:
fc_trajectory = TrajectoryFC(rho=0.3, eta=0.1)
fc_trajectory.set_initial_state(np.array([x0, y0, 1.0, 0.0]))
fc_system = SystemFC(potential, fc_trajectory)
fc_solution = fc_system.simulate(
    step=0.01,
    t_span=(0.0, 0.2),
    n_save_step=11,
    check_energy=True,
    progress=False,
)
fc_energy = generalized_energy(fc_system, fc_solution)[0]
fc_scale = max(abs(fc_energy[0]), np.finfo(float).eps)
fc_relative_error = (fc_energy - fc_energy[0]) / fc_scale

print(
    f"FC: n_steps={fc_solution.n_steps}, err={fc_solution.err:.3e}, "
    f"max |error rel.|={np.max(np.abs(fc_relative_error)):.3e}"
)